# 04 — DistilBERT Prosodic Boundary Classifier

Fine-tunes `distilbert-base-uncased` on the silver-standard labels produced by
`annotation_pipeline_v4.ipynb`.

**Task (current):** Binary token classification — predict whether each word is a
prosodic phrase boundary (1) or not (0).

**Baseline to beat:** PSST text-only GPT-Neo F1 = 0.77

Disclaimer: Code was largely generated with the help of Claude Sonnet 3.5 (Anthropic, 2026 February - May). Prompts, code tweaks and verification by me.

---

## How to use this notebook

Run cells **top to bottom**. The only cell you should need to edit is **Cell 1**.

| Section | Cells | What it does |
|---|---|---|
| **1 · Configuration** | 1 | All hyperparameters, flags, and paths |
| **2 · Environment** | 2 | Mount Drive, install packages |
| **3 · Imports** | 3 | All imports + seed setting |
| **4 · Load Data** | 4 | Read label files from Drive |
| **5 · Dataset** | 5 | `ProsodyDataset`, label alignment, punctuation toggle |
| **6 · Split** | 6 | 80/10/10 split + DataLoaders |
| **7 · Model** | 7 | `ProsodyBoundaryModel` with extensible head hooks |
| **8 · Metrics** | 8 | `flatten_predictions`, `compute_metrics` |
| **9 · Training** | 9 | Training loop with per-epoch logging |
| **10 · Evaluation** | 10 | Test-set eval, loss/F1 curves, confusion matrix |
| **11 · Save** | 11 | Checkpoint, `hparams.json`, predictions, plots |

---

## Output folder structure

```
runs/
└── run_{timestamp}_{config_hash}/
    ├── checkpoint/          ← model weights + tokenizer (HuggingFace format)
    ├── hparams.json         ← ALL hyperparameters (model + annotation tool)
    ├── test_predictions.json  ← per-sample word-level predictions
    ├── curves.png           ← loss + F1 learning curves
    └── confusion_matrix.png
```

---

## Adding extra input features

The architecture is built to be expanded.  To add a feature (e.g. POS tags):
1. Add its name to `EXTRA_FEATURES` in **Cell 1**.
2. Implement a builder function in **Cell 5** (`ProsodyDataset._process`).
3. Embed + concatenate in **Cell 7** (`ProsodyBoundaryModel.forward`).
4. Add it to `prosody_collate_fn` in **Cell 5**.
Stub comments mark each extension point.


---
## Section 1 · Configuration

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 1 — CONFIGURATION                                      ║
# ║  The only cell you should need to edit between runs.         ║
# ╚══════════════════════════════════════════════════════════════╝
import os, json, hashlib
from datetime import datetime

# ── Paths ─────────────────────────────────────────────────────────────────────
DRIVE_ROOT  = "/content/drive/MyDrive/Capstone"
LABELS_ROOT = f"{DRIVE_ROOT}/labels"
RUNS_ROOT   = f"{DRIVE_ROOT}/runs"
BATCHED_ROOT = f"{DRIVE_ROOT}/labels_batched"

# ── Data split ────────────────────────────────────────────────────────────────
SPLIT_SEED = 42
TRAIN_FRAC = 0.80
VAL_FRAC   = 0.10

# ── Text pre-processing ───────────────────────────────────────────────────────
STRIP_PUNCTUATION = True
EXTRA_FEATURES    = []

# ── Model ─────────────────────────────────────────────────────────────────────
MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 128

# ── Training ──────────────────────────────────────────────────────────────────
BATCH_SIZE    = 16
LEARNING_RATE = 2e-5
NUM_EPOCHS    = 10
WARMUP_STEPS  = 0
WEIGHT_DECAY  = 0.01

# ── Class-imbalance strategy (boundary head only) ─────────────────────────────
IMBALANCE_STRATEGY   = "none"
BOUNDARY_LOSS_WEIGHT = 5.0

# ── Multi-task loss weights ───────────────────────────────────────────────────
# Weight for each head's contribution to the combined loss.
# 1.0 = equal weighting. Raise a weight to emphasise that head's gradient.
# Intonation and break_idx heads supervise only at boundary positions, so
# their raw loss magnitude will naturally be smaller than the boundary head's.
BOUNDARY_TASK_WEIGHT   = 1.0
INTONATION_TASK_WEIGHT = 1.0
BREAK_IDX_TASK_WEIGHT  = 1.0

# ── Run metadata ──────────────────────────────────────────────────────────────
RUN_NOTES = ""

# ── Auto-generated run ID (do not edit) ──────────────────────────────────────
_cfg = dict(
    SPLIT_SEED=SPLIT_SEED, TRAIN_FRAC=TRAIN_FRAC, VAL_FRAC=VAL_FRAC,
    STRIP_PUNCTUATION=STRIP_PUNCTUATION, EXTRA_FEATURES=EXTRA_FEATURES,
    MODEL_NAME=MODEL_NAME, MAX_LENGTH=MAX_LENGTH,
    BATCH_SIZE=BATCH_SIZE, LEARNING_RATE=LEARNING_RATE,
    NUM_EPOCHS=NUM_EPOCHS, WARMUP_STEPS=WARMUP_STEPS, WEIGHT_DECAY=WEIGHT_DECAY,
    IMBALANCE_STRATEGY=IMBALANCE_STRATEGY, BOUNDARY_LOSS_WEIGHT=BOUNDARY_LOSS_WEIGHT,
    BOUNDARY_TASK_WEIGHT=BOUNDARY_TASK_WEIGHT,
    INTONATION_TASK_WEIGHT=INTONATION_TASK_WEIGHT,
    BREAK_IDX_TASK_WEIGHT=BREAK_IDX_TASK_WEIGHT,
)
_cfg_hash = hashlib.md5(json.dumps(_cfg, sort_keys=True).encode()).hexdigest()[:8]
RUN_ID    = f"run_{datetime.now().strftime('%Y%m%d_%H%M%S')}_{_cfg_hash}"
RUN_DIR   = f"{RUNS_ROOT}/{RUN_ID}"

print("✓ Configuration loaded.")
print(f"  Run ID:               {RUN_ID}")
print(f"  Labels root:          {LABELS_ROOT}")
print(f"  Strip punctuation:    {STRIP_PUNCTUATION}")
print(f"  Extra features:       {EXTRA_FEATURES or 'none'}")
print(f"  Imbalance strategy:   {IMBALANCE_STRATEGY}")
print(f"  Epochs / LR / Batch:  {NUM_EPOCHS} / {LEARNING_RATE} / {BATCH_SIZE}")
print(f"  Task weights:         boundary={BOUNDARY_TASK_WEIGHT}  "
      f"intonation={INTONATION_TASK_WEIGHT}  break_idx={BREAK_IDX_TASK_WEIGHT}")

---
## Section 2 · Environment Setup

Run once per Colab session. Safe to re-run — all steps are idempotent.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 2 — Mount Drive + install packages                     ║
# ╚══════════════════════════════════════════════════════════════╝
from google.colab import drive
import os

drive.mount("/content/drive", force_remount=True)

for d in [LABELS_ROOT, RUNS_ROOT]:
    os.makedirs(d, exist_ok=True)

!pip install -q transformers datasets scikit-learn matplotlib

import torch
print(f"\n✓ PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    print("⚠  No GPU — training will be slow.")
    print("   Runtime → Change runtime type → T4 GPU")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"   Device: {device}")


---
## Section 3 · Imports

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 3 — Imports                                            ║
# ╚══════════════════════════════════════════════════════════════╝
import os, json, string, random, shutil, traceback
from datetime import datetime

import numpy as np
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader, Subset
import matplotlib.pyplot as plt
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    confusion_matrix, ConfusionMatrixDisplay,
)
from transformers import (
    DistilBertTokenizerFast,
    DistilBertModel,
    DistilBertPreTrainedModel,
    DistilBertConfig,
    get_linear_schedule_with_warmup,
)

# JSON encoder that handles numpy scalar types produced by sklearn metrics.
# Used in Cell 11 when saving hparams.json.
class _NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.floating): return float(obj)
        if isinstance(obj, np.integer):  return int(obj)
        if isinstance(obj, np.ndarray):  return obj.tolist()
        return super().default(obj)

# Reproducibility
torch.manual_seed(SPLIT_SEED)
np.random.seed(SPLIT_SEED)
random.seed(SPLIT_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SPLIT_SEED)

print("✓ Imports complete.")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 3b — Consolidate label files into batches (run once)   ║
# ╚══════════════════════════════════════════════════════════════╝
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm
import json, os, time

FILE_BATCH_SIZE = 1000

b_dir = os.path.join(LABELS_ROOT, "boundary")
i_dir = os.path.join(LABELS_ROOT, "intonation")
x_dir = os.path.join(LABELS_ROOT, "break_idx")

if os.path.exists(BATCHED_ROOT) and any(
    f.startswith("batch_") for f in os.listdir(BATCHED_ROOT)
):
    print(f"✓ Batched files already exist at {BATCHED_ROOT} — skipping consolidation.")
    print(f"  Delete {BATCHED_ROOT} to force a rebuild.")
else:
    # ── Retry-safe listdir ───────────────────────────────────────
    def retry_listdir(path, retries=5, delay=10):
        for attempt in range(retries):
            try:
                return os.listdir(path)
            except Exception as e:
                if attempt < retries - 1:
                    print(f"  ⚠ listdir({path}) failed ({e}) — retrying in {delay}s ({attempt+1}/{retries})...")
                    time.sleep(delay)
                else:
                    raise

    print("Scanning label directories...", flush=True)
    b_ids = {f[:-5] for f in retry_listdir(b_dir) if f.endswith(".json") and f != "meta.json"}
    i_ids = {f[:-5] for f in retry_listdir(i_dir) if f.endswith(".json") and f != "meta.json"}
    x_ids = {f[:-5] for f in retry_listdir(x_dir) if f.endswith(".json") and f != "meta.json"}
    sample_ids = sorted(b_ids & i_ids & x_ids)
    n_batches  = (len(sample_ids) + BATCH_SIZE - 1) // BATCH_SIZE
    print(f"  {len(sample_ids):,} matched samples → {n_batches} batches of {BATCH_SIZE}")

    # ── Parallel read into RAM ────────────────────────────────────
    def load_one(sid):
        try:
            with open(os.path.join(b_dir, f"{sid}.json")) as f: b = json.load(f)
            with open(os.path.join(i_dir, f"{sid}.json")) as f: i = json.load(f)
            with open(os.path.join(x_dir, f"{sid}.json")) as f: x = json.load(f)
            return sid, b, i, x, None
        except Exception as e:
            return sid, None, None, None, str(e)

    all_data = {}
    with ThreadPoolExecutor(max_workers=32) as executor:
        futures = {executor.submit(load_one, sid): sid for sid in sample_ids}
        for future in tqdm(as_completed(futures), total=len(sample_ids), desc="Reading from Drive", unit="file"):
            sid, b, i, x, err = future.result()
            if err:
                tqdm.write(f"  ⚠ {sid}: {err}")
            else:
                all_data[sid] = {"b": b, "i": i, "x": x}

    # ── Write batch files to Drive ────────────────────────────────
    os.makedirs(BATCHED_ROOT, exist_ok=True)
    batches = [sample_ids[k:k + FILE_BATCH_SIZE] for k in range(0, len(sample_ids), BATCH_SIZE)]
    for idx, batch in enumerate(tqdm(batches, desc="Writing batches to Drive", unit="batch")):
        out = {sid: all_data[sid] for sid in batch if sid in all_data}
        with open(os.path.join(BATCHED_ROOT, f"batch_{idx:04d}.json"), "w") as f:
            json.dump(out, f)

    print(f"\n✓ Written {len(batches)} batch files to {BATCHED_ROOT}")
    print(f"  This is a one-time step. Future sessions skip directly to Cell 4.")

---
## Section 4 · Load Label Files

Reads `boundary/`, `intonation/`, and `break_idx/` directly from Drive.

**Label file schema (from `annotation_pipeline_v4`):**

| Directory | Key used | Type | Description |
|---|---|---|---|
| `boundary/{id}.json` | `consensus` | `list[int]` 0/1 | Training target — both annotators agreed |
| `boundary/{id}.json` | `psst` | `list[int]` | Raw PSST boundary vector (provenance) |
| `boundary/{id}.json` | `wav2tobi` | `list[int]` | Raw Wav2ToBI boundary vector (provenance) |
| `boundary/{id}.json` | `status` | `list[str]` | Per-token: `agree_boundary` / `agree_none` / `psst_only` / `wav2tobi_only` |
| `intonation/{id}.json` | `labels` | `list[int]` | 0=none, 1=rising, 2=falling, 3=level, 4=unclear |
| `break_idx/{id}.json` | `labels` | `list[str]` | `"3"` / `"4"` / `""` |

Each directory also contains a `meta.json` (tool hyperparameters) that is
excluded during file enumeration and collected separately for provenance
tracking in `hparams.json`.


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 4 — Load label files from batched Drive files          ║
# ╚══════════════════════════════════════════════════════════════╝
from tqdm.notebook import tqdm

def load_label_files_batched(batched_root):
    """
    Load all samples from the consolidated batch files produced by Cell 3b.

    Batch file layout:
        labels_batched/
        ├── batch_0000.json   ← dict keyed by sample_id, each value has "b", "i", "x"
        ├── batch_0001.json
        └── ...

    Returns
    -------
    samples  : list[dict]  — keys: sample_id, tokens, boundary_labels,
                             intonation_labels, break_idx_labels
    meta     : dict        — empty (meta.json not included in batches;
                             load separately from LABELS_ROOT if needed)
    """
    batch_files = sorted(
        f for f in os.listdir(batched_root)
        if f.startswith("batch_") and f.endswith(".json")
    )
    if not batch_files:
        raise FileNotFoundError(
            f"No batch files found in {batched_root}.\n"
            "Run Cell 3b first to consolidate the label files."
        )
    print(f"Found {len(batch_files)} batch files in {batched_root}.")

    samples, n_skipped = [], 0
    for fname in tqdm(batch_files, desc="Loading batches", unit="batch"):
        with open(os.path.join(batched_root, fname)) as f:
            batch = json.load(f)
        for sid, data in batch.items():
            b, i, x = data["b"], data["i"], data["x"]
            n = len(b["tokens"])
            if not (len(b["consensus"]) == len(i["labels"]) == len(x["labels"]) == n):
                print(f"  ⚠ {sid}: mismatched label lengths — skipping.")
                n_skipped += 1
                continue
            samples.append({
                "sample_id":         sid,
                "tokens":            b["tokens"],
                "boundary_labels":   b["consensus"],   # 0/1 — silver-standard training target
                "intonation_labels": i["labels"],       # int  0–4
                "break_idx_labels":  x["labels"],       # str  "3" / "4" / ""
            })

    print(f"\n✓ Loaded {len(samples):,} samples  ({n_skipped} skipped).")
    if samples:
        total_tokens     = sum(len(s["tokens"])          for s in samples)
        total_boundaries = sum(sum(s["boundary_labels"]) for s in samples)
        pos_rate         = total_boundaries / max(total_tokens, 1)
        ratio            = (1 - pos_rate) / max(pos_rate, 1e-9)
        print(f"  Total words:     {total_tokens:,}")
        print(f"  Boundary tokens: {total_boundaries:,}  ({100*pos_rate:.1f}% positive rate)")
        print(f"  Non-boundary:boundary ratio ≈ {ratio:.1f}:1")
        print(f"  → Suggested BOUNDARY_LOSS_WEIGHT: ~{round(ratio)}.0")

    # meta.json provenance is not bundled into batch files.
    # Load separately if needed: open(LABELS_ROOT/boundary/meta.json) etc.
    return samples, {}


all_samples, label_meta = load_label_files_batched(BATCHED_ROOT)

---
## Section 5 · Dataset

`ProsodyDataset` tokenizes each sentence with DistilBERT and aligns word-level
boundary labels to sub-word tokens using the standard HuggingFace
`tokenize_and_align_labels` pattern:

- **First sub-word** of word *i* → receives the word's label
- **Continuation sub-words** → label set to `-100` (ignored by loss)
- **Special tokens** (`[CLS]`, `[SEP]`, `[PAD]`) → `-100`


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 5 — ProsodyDataset + collate_fn                        ║
# ╚══════════════════════════════════════════════════════════════╝

PUNCT_SET = set(string.punctuation)

def is_punct_only(word):
    """True if every character in `word` is ASCII punctuation."""
    return bool(word) and all(c in PUNCT_SET for c in word)


class ProsodyDataset(Dataset):
    """
    Token-classification dataset for prosodic boundary detection.

    Label alignment rules
    ─────────────────────
    All three heads share the same rule: first sub-word of word i receives
    the word's label; continuation sub-words and special tokens get -100.

    Boundary   : 0 / 1  — all positions supervised
    Intonation : rising(1)→0 | falling(2)→1 | level(3)→2
                 none(0) and unclear(4) → -100  (masked)
    Break index: "3"→0 | "4"→1
                 ""  and any missing value → -100 (masked, per Opus:
                 "" means Wav2ToBI was never asked, not an affirmative
                 "no break" — fabricating class-0 here would be a label
                 fidelity error)
    """
    SPK_CHANGE_TOKEN = "/"   # must match the pipeline constant

    def __init__(self, samples, tokenizer, max_length=128,
                 strip_punctuation=False, extra_features=None):
        self.tokenizer         = tokenizer
        self.max_length        = max_length
        self.strip_punctuation = strip_punctuation
        self.extra_features    = extra_features or []
        self.items             = [self._process(s) for s in samples]

    def _process(self, sample):
        words    = list(sample["tokens"])
        b_labels = list(sample["boundary_labels"])
        i_labels = list(sample.get("intonation_labels", []))
        x_labels = list(sample.get("break_idx_labels",  []))

        # ── Optional: strip punctuation-only tokens ───────────────────────
        if self.strip_punctuation:
            quads = [
                (w, b, i, x)
                for w, b, i, x in zip(words, b_labels, i_labels, x_labels)
                if not is_punct_only(w) or w == self.SPK_CHANGE_TOKEN
            ]
            if quads:
                words, b_labels, i_labels, x_labels = map(list, zip(*quads))
            else:
                words, b_labels, i_labels, x_labels = [], [], [], []

        # ── Sub-word tokenization ─────────────────────────────────────────
        encoding = self.tokenizer(
            words,
            is_split_into_words=True,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

        word_ids = encoding.word_ids(batch_index=0)

        # ── Align boundary labels ─────────────────────────────────────────
        aligned_boundary = []
        prev_word_i = None
        for word_i in word_ids:
            if word_i is None:
                aligned_boundary.append(-100)
            elif word_i != prev_word_i:
                aligned_boundary.append(
                    b_labels[word_i] if word_i < len(b_labels) else -100)
            else:
                aligned_boundary.append(-100)
            prev_word_i = word_i

        # ── Speaker-change position mask ──────────────────────────────────────
        # True at the first sub-word of every "/" token.R
        # Used at eval time to exclude these from boundary F1.
        aligned_spk_mask = []
        prev_word_i = None
        for word_i in word_ids:
            if word_i is None:
                aligned_spk_mask.append(False)
            elif word_i != prev_word_i:
                aligned_spk_mask.append(words[word_i] == self.SPK_CHANGE_TOKEN)
            else:
                aligned_spk_mask.append(False)
            prev_word_i = word_i

        # ── Align intonation labels ───────────────────────────────────────
        # Raw: 0=none, 1=rising, 2=falling, 3=level, 4=unclear
        # Mapped: rising→0, falling→1, level→2; none+unclear → -100
        _INTONATION_MAP = {1: 0, 2: 1, 3: 2}
        aligned_intonation = []
        prev_word_i = None
        for word_i in word_ids:
            if word_i is None:
                aligned_intonation.append(-100)
            elif word_i != prev_word_i:
                raw = i_labels[word_i] if word_i < len(i_labels) else -1
                aligned_intonation.append(_INTONATION_MAP.get(raw, -100))
            else:
                aligned_intonation.append(-100)
            prev_word_i = word_i

        # ── Align break index labels ──────────────────────────────────────
        # Raw: "" (unannotated), "3" (intermediate phrase), "4" (intonational phrase)
        # Mapped: "3"→0, "4"→1; "" and missing → -100
        # "" is masked rather than treated as class-0 because Wav2ToBI never
        # produces an affirmative "no break" label — the absence of annotation
        # is not the same as a negative example.
        _BREAK_IDX_MAP = {"3": 0, "4": 1}
        aligned_break_idx = []
        prev_word_i = None
        for word_i in word_ids:
            if word_i is None:
                aligned_break_idx.append(-100)
            elif word_i != prev_word_i:
                raw = x_labels[word_i] if word_i < len(x_labels) else ""
                aligned_break_idx.append(_BREAK_IDX_MAP.get(raw, -100))
            else:
                aligned_break_idx.append(-100)
            prev_word_i = word_i

        # ── Extra feature hooks ───────────────────────────────────────────
        extra = {}
        # if "pos" in self.extra_features:
        #     extra["pos_ids"] = build_pos_ids(words, encoding, self.max_length)

        return {
            "sample_id":         sample["sample_id"],
            "tokens":            words,   # post-strip; aligned with label tensors
            "input_ids":         encoding["input_ids"].squeeze(0),
            "attention_mask":    encoding["attention_mask"].squeeze(0),
            "spk_mask": torch.tensor(aligned_spk_mask, dtype=torch.bool),
            "labels":            torch.tensor(aligned_boundary,   dtype=torch.long),
            "intonation_labels": torch.tensor(aligned_intonation, dtype=torch.long),
            "break_idx_labels":  torch.tensor(aligned_break_idx,  dtype=torch.long),
            **extra,
        }

    def __len__(self):  return len(self.items)
    def __getitem__(self, idx): return self.items[idx]


def prosody_collate_fn(batch):
    """Custom collate: stack tensors, keep sample_ids + tokens as plain lists."""
    out = {
        "input_ids":         torch.stack([b["input_ids"]         for b in batch]),
        "attention_mask":    torch.stack([b["attention_mask"]    for b in batch]),
        "spk_mask":          torch.stack([item["spk_mask"] for item in batch]),
        "labels":            torch.stack([b["labels"]            for b in batch]),
        "intonation_labels": torch.stack([b["intonation_labels"] for b in batch]),
        "break_idx_labels":  torch.stack([b["break_idx_labels"]  for b in batch]),
        "sample_ids":        [b["sample_id"] for b in batch],
        "tokens":            [b["tokens"]    for b in batch],
    }
    # if "pos_ids" in batch[0]:
    #     out["pos_ids"] = torch.stack([b["pos_ids"] for b in batch])
    return out


print("✓ ProsodyDataset and prosody_collate_fn defined.")

---
## Section 6 · Train / Val / Test Split

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 6 — Tokenizer, split, DataLoaders                      ║
# ╚══════════════════════════════════════════════════════════════╝

assert all_samples, "No samples loaded — run Cell 4 first."

tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

full_dataset = ProsodyDataset(
    all_samples, tokenizer,
    max_length=MAX_LENGTH,
    strip_punctuation=STRIP_PUNCTUATION,
    extra_features=EXTRA_FEATURES,
)

# ── Deterministic 80 / 10 / 10 split ─────────────────────────────────────────
rng     = random.Random(SPLIT_SEED)
indices = list(range(len(full_dataset)))
rng.shuffle(indices)

n_train   = int(len(indices) * TRAIN_FRAC)
n_val     = int(len(indices) * VAL_FRAC)
train_idx = indices[:n_train]
val_idx   = indices[n_train : n_train + n_val]
test_idx  = indices[n_train + n_val :]

train_ds = Subset(full_dataset, train_idx)
val_ds   = Subset(full_dataset, val_idx)
test_ds  = Subset(full_dataset, test_idx)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=prosody_collate_fn)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=prosody_collate_fn)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=prosody_collate_fn)

print(f"✓ Dataset split  (seed={SPLIT_SEED})")
print(f"  Train: {len(train_ds):>4} samples  ({len(train_loader)} batch{'es' if len(train_loader)!=1 else ''})")
print(f"  Val:   {len(val_ds):>4} samples  ({len(val_loader)} batch{'es' if len(val_loader)!=1 else ''})")
print(f"  Test:  {len(test_ds):>4} samples  ({len(test_loader)} batch{'es' if len(test_loader)!=1 else ''})")


---
## Section 7 · Model

`ProsodyBoundaryModel` wraps DistilBERT with a single linear head for binary
boundary classification.  The `forward()` signature accepts an optional
`loss_weights` tensor so the imbalance strategy is controlled at call-time,
not baked into the model.

Hooks for `intonation_head` and `break_idx_head` are left in comments for
when the multi-head extension is ready.


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 7 — ProsodyBoundaryModel                               ║
# ╚══════════════════════════════════════════════════════════════╝

class ProsodyBoundaryModel(DistilBertPreTrainedModel):
    """
    Architecture
    ────────────
    DistilBERT encoder
        └─► dropout (seq_classif_dropout)
             ├─► boundary_head    Linear(H → 2)   all positions
             ├─► intonation_head  Linear(H → 3)   rising / falling / level
             └─► break_idx_head   Linear(H → 2)   index-3 / index-4

    Each head returns logits over all (B, T) positions; masking to the
    relevant subset is handled by CrossEntropyLoss(ignore_index=-100).
    Loss combination weights are controlled by BOUNDARY/INTONATION/
    BREAK_IDX_TASK_WEIGHT in Cell 1 and applied in the training loop.
    """

    def __init__(self, config):
        super().__init__(config)
        self.distilbert      = DistilBertModel(config)
        self.dropout         = nn.Dropout(config.seq_classif_dropout)
        self.boundary_head   = nn.Linear(config.hidden_size, 2)
        self.intonation_head = nn.Linear(config.hidden_size, 3)  # rising/falling/level
        self.break_idx_head  = nn.Linear(config.hidden_size, 2)  # index-3/index-4
        self.post_init()

    def forward(self, input_ids, attention_mask,
                labels=None, intonation_labels=None, break_idx_labels=None,
                loss_weights=None):
        """
        Parameters
        ----------
        input_ids          : (B, T)
        attention_mask     : (B, T)
        labels             : (B, T)  boundary targets; -100 at ignored positions
        intonation_labels  : (B, T)  0/1/2 at boundary positions, -100 elsewhere
        break_idx_labels   : (B, T)  0/1   at boundary positions, -100 elsewhere
        loss_weights       : (2,) FloatTensor or None — boundary class weights only

        Returns
        -------
        dict with keys:
            losses             : dict  {head_name: scalar loss tensor}
                                 only present when at least one label tensor is passed
            boundary_logits    : (B, T, 2)
            intonation_logits  : (B, T, 3)
            break_idx_logits   : (B, T, 2)
        """
        outputs = self.distilbert(input_ids=input_ids,
                                  attention_mask=attention_mask)
        seq_out = self.dropout(outputs.last_hidden_state)   # (B, T, H)

        boundary_logits   = self.boundary_head(seq_out)     # (B, T, 2)
        intonation_logits = self.intonation_head(seq_out)   # (B, T, 3)
        break_idx_logits  = self.break_idx_head(seq_out)    # (B, T, 2)

        losses = {}
        if labels is not None:
           losses["boundary"] = nn.CrossEntropyLoss(
              weight=loss_weights, ignore_index=-100)(
              boundary_logits.view(-1, 2), labels.view(-1))

        if intonation_labels is not None and (intonation_labels != -100).any():
            losses["intonation"] = nn.CrossEntropyLoss(ignore_index=-100)(
               intonation_logits.view(-1, 3), intonation_labels.view(-1))

        if break_idx_labels is not None and (break_idx_labels != -100).any():
            losses["break_idx"] = nn.CrossEntropyLoss(ignore_index=-100)(
                break_idx_logits.view(-1, 2), break_idx_labels.view(-1))

        out = {
            "boundary_logits":   boundary_logits,
            "intonation_logits": intonation_logits,
            "break_idx_logits":  break_idx_logits,
        }
        if losses:
            out["losses"] = losses
        return out

@classmethod
def _can_set_experts_implementation(cls):
    return False

ProsodyBoundaryModel._can_set_experts_implementation = _can_set_experts_implementation

def build_model():
    cfg = DistilBertConfig.from_pretrained(MODEL_NAME,
                                           num_labels=2,
                                           seq_classif_dropout=0.2)
    model = ProsodyBoundaryModel.from_pretrained(MODEL_NAME, config=cfg,
                                                  ignore_mismatched_sizes=True,
                                                  _fast_init=False)
    model.gradient_checkpointing_enable()   # ← add this
    return model.to(device)

model = build_model()
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✓ ProsodyBoundaryModel ready  ({n_params:,} trainable parameters)")
print(f"  Heads: boundary(2)  intonation(3)  break_idx(2)")

---
## Section 8 · Metrics Helpers

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 8 — Metrics helpers                                    ║
# ╚══════════════════════════════════════════════════════════════╝

def flatten_predictions(all_logits, all_labels, all_spk_masks=None):
    """
    Convert batched (logits, labels) to flat numpy arrays,
    discarding all positions where label == -100. Optionally discards
    speaker-change token positions (spk_mask == True)

    Parameters
    ----------
    all_logits : list of (B, T, C) tensors  — works for any C
    all_labels : list of (B, T)   tensors

    Returns
    -------
    preds  : np.ndarray (N,)
    labels : np.ndarray (N,)
    """
    flat_preds, flat_labels = [], []
    for idx, (logits_batch, labels_batch) in enumerate(zip(all_logits, all_labels)):
        preds_batch = logits_batch.argmax(dim=-1)   # (B, T)
        spk_batch   = all_spk_masks[idx] if all_spk_masks else None
        for b_i, (preds_seq, labels_seq) in enumerate(zip(preds_batch, labels_batch)):
            mask = labels_seq != -100
            if spk_batch is not None:
                mask = mask & ~spk_batch[b_i]       # exclude speaker-change positions
            flat_preds.extend(preds_seq[mask].cpu().tolist())
            flat_labels.extend(labels_seq[mask].cpu().tolist())
    return np.array(flat_preds), np.array(flat_labels)


def compute_metrics(flat_preds, flat_labels):
    """Binary P/R/F1 for the boundary class (pos_label=1)."""
    return {
        "f1":        f1_score(flat_labels,        flat_preds, pos_label=1, zero_division=0),
        "precision": precision_score(flat_labels, flat_preds, pos_label=1, zero_division=0),
        "recall":    recall_score(flat_labels,    flat_preds, pos_label=1, zero_division=0),
    }


def compute_multiclass_metrics(flat_preds, flat_labels):
    """Macro-averaged P/R/F1 for multi-class heads (intonation, break_idx)."""
    return {
        "f1":        f1_score(flat_labels,        flat_preds, average="macro", zero_division=0),
        "precision": precision_score(flat_labels, flat_preds, average="macro", zero_division=0),
        "recall":    recall_score(flat_labels,    flat_preds, average="macro", zero_division=0),
    }


print("✓ Metrics helpers defined.")

In [ ]:
'''
print(f"GPU allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"GPU reserved:  {torch.cuda.memory_reserved()/1e9:.2f} GB")
print(torch.cuda.memory_summary(abbreviated=True))
'''

---
## Section 9 · Training Loop

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 9 — Training loop                                      ║
# ╚══════════════════════════════════════════════════════════════╝

from tqdm.notebook import tqdm as tqdm_nb

# ── Boundary class-imbalance weights ─────────────────────────────────────────
if IMBALANCE_STRATEGY == "weighted_loss":
    loss_weights = torch.tensor([1.0, BOUNDARY_LOSS_WEIGHT],
                                dtype=torch.float).to(device)
    print(f"ℹ  Weighted loss  [non-boundary=1.0, boundary={BOUNDARY_LOSS_WEIGHT}]")
else:
    loss_weights = None
    print("ℹ  No class-imbalance correction  (IMBALANCE_STRATEGY='none')")

# ── Optimizer + scheduler ─────────────────────────────────────────────────────
optimizer   = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
total_steps = NUM_EPOCHS * len(train_loader)
scheduler   = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=WARMUP_STEPS, num_training_steps=total_steps)

# ── History ───────────────────────────────────────────────────────────────────
history = {
    "train_loss": [], "val_loss":  [],
    # Boundary — primary metric; track P/R/F1
    "train_boundary_f1":   [], "val_boundary_f1":   [],
    "train_boundary_prec": [], "val_boundary_prec": [],
    "train_boundary_rec":  [], "val_boundary_rec":  [],
    # Intonation + break index — macro F1 only
    "train_intonation_f1": [], "val_intonation_f1": [],
    "train_break_idx_f1":  [], "val_break_idx_f1":  [],
}
best_val_f1 = -1.0   # tracked on boundary F1 (primary task)
best_epoch  = -1
best_state  = None

print(f"\nStarting training: {NUM_EPOCHS} epochs  |  "
      f"{len(train_loader)} steps/epoch  |  {total_steps} total optimizer steps")
print("=" * 70)

for epoch in tqdm_nb(range(1, NUM_EPOCHS + 1), desc = "Epochs"):

    # ── Train ─────────────────────────────────────────────────────────────
    model.train()
    train_loss = 0.0
    # Separate logit/label collectors per head
    t_b_logits, t_b_labels = [], []
    t_i_logits, t_i_labels = [], []
    t_x_logits, t_x_labels = [], []
    t_b_spk_masks = []

    for batch in tqdm_nb(train_loader, desc=f"Ep {epoch:02d} train", leave=False):
        input_ids         = batch["input_ids"].to(device)
        attention_mask    = batch["attention_mask"].to(device)
        labels            = batch["labels"].to(device)
        intonation_labels = batch["intonation_labels"].to(device)
        break_idx_labels  = batch["break_idx_labels"].to(device)

        optimizer.zero_grad()
        out = model(input_ids, attention_mask,
                    labels=labels,
                    intonation_labels=intonation_labels,
                    break_idx_labels=break_idx_labels,
                    loss_weights=loss_weights)

        # Weighted combination of per-head losses
        losses = out["losses"]
        loss = sum(
            w * losses[k]
            for k, w in [("boundary",   BOUNDARY_TASK_WEIGHT),
                         ("intonation", INTONATION_TASK_WEIGHT),
                         ("break_idx",  BREAK_IDX_TASK_WEIGHT)]
            if k in losses
        )
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        train_loss += loss.item()
        t_b_logits.append(out["boundary_logits"].detach().cpu())
        t_b_labels.append(labels.detach().cpu())
        t_i_logits.append(out["intonation_logits"].detach().cpu())
        t_i_labels.append(intonation_labels.detach().cpu())
        t_x_logits.append(out["break_idx_logits"].detach().cpu())
        t_x_labels.append(break_idx_labels.detach().cpu())
        t_b_spk_masks.append(batch["spk_mask"].cpu())

    train_loss /= len(train_loader)
    t_b_preds, t_b_true = flatten_predictions(t_b_logits, t_b_labels, all_spk_masks=t_b_spk_masks)
    t_i_preds, t_i_true = flatten_predictions(t_i_logits, t_i_labels)
    t_x_preds, t_x_true = flatten_predictions(t_x_logits, t_x_labels)
    t_bm = compute_metrics(t_b_preds, t_b_true)
    t_im = compute_multiclass_metrics(t_i_preds, t_i_true)
    t_xm = compute_multiclass_metrics(t_x_preds, t_x_true)

    # ── Validate ──────────────────────────────────────────────────────────
    model.eval()
    val_loss = 0.0
    v_b_logits, v_b_labels = [], []
    v_i_logits, v_i_labels = [], []
    v_x_logits, v_x_labels = [], []
    v_b_spk_masks = []

    with torch.no_grad():
        for batch in val_loader:
            input_ids         = batch["input_ids"].to(device)
            attention_mask    = batch["attention_mask"].to(device)
            labels            = batch["labels"].to(device)
            intonation_labels = batch["intonation_labels"].to(device)
            break_idx_labels  = batch["break_idx_labels"].to(device)

            out = model(input_ids, attention_mask,
                        labels=labels,
                        intonation_labels=intonation_labels,
                        break_idx_labels=break_idx_labels,
                        loss_weights=loss_weights)

            losses = out["losses"]
            loss = sum(
                w * losses[k]
                for k, w in [("boundary",   BOUNDARY_TASK_WEIGHT),
                             ("intonation", INTONATION_TASK_WEIGHT),
                             ("break_idx",  BREAK_IDX_TASK_WEIGHT)]
                if k in losses
            )
            val_loss += loss.item()
            v_b_logits.append(out["boundary_logits"].cpu())
            v_b_labels.append(labels.cpu())
            v_i_logits.append(out["intonation_logits"].cpu())
            v_i_labels.append(intonation_labels.cpu())
            v_x_logits.append(out["break_idx_logits"].cpu())
            v_x_labels.append(break_idx_labels.cpu())

    val_loss /= max(len(val_loader), 1)
    v_b_preds, v_b_true = flatten_predictions(v_b_logits, v_b_labels)
    v_i_preds, v_i_true = flatten_predictions(v_i_logits, v_i_labels)
    v_x_preds, v_x_true = flatten_predictions(v_x_logits, v_x_labels)
    v_bm = compute_metrics(v_b_preds, v_b_true)
    v_im = compute_multiclass_metrics(v_i_preds, v_i_true)
    v_xm = compute_multiclass_metrics(v_x_preds, v_x_true)

    # ── Log ───────────────────────────────────────────────────────────────
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_boundary_f1"].append(t_bm["f1"])
    history["val_boundary_f1"].append(v_bm["f1"])
    history["train_boundary_prec"].append(t_bm["precision"])
    history["val_boundary_prec"].append(v_bm["precision"])
    history["train_boundary_rec"].append(t_bm["recall"])
    history["val_boundary_rec"].append(v_bm["recall"])
    history["train_intonation_f1"].append(t_im["f1"])
    history["val_intonation_f1"].append(v_im["f1"])
    history["train_break_idx_f1"].append(t_xm["f1"])
    history["val_break_idx_f1"].append(v_xm["f1"])

    # Best checkpoint tracked on boundary F1 (primary task)
    star = "★" if v_bm["f1"] > best_val_f1 else " "
    print(f"Ep {epoch:02d}/{NUM_EPOCHS}  loss={train_loss:.4f}→{val_loss:.4f}  "
          f"| boundary F1={t_bm['f1']:.4f}→{v_bm['f1']:.4f} {star}"
          f"| inton F1={v_im['f1']:.4f}  break F1={v_xm['f1']:.4f}")

    if v_bm["f1"] > best_val_f1:
        best_val_f1 = v_bm["f1"]
        best_epoch  = epoch
        best_state  = {k: v.cpu().clone() for k, v in model.state_dict().items()}

print("=" * 70)
print(f"Training complete.  Best val boundary F1 = {best_val_f1:.4f}  (epoch {best_epoch})")

---
## Section 10 · Evaluation on Test Set

Reloads the best checkpoint (by val F1) and evaluates on the held-out test set.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 10 — Test-set evaluation + diagnostic plots            ║
# ╚══════════════════════════════════════════════════════════════╝

assert best_state is not None, "No checkpoint found — run Cell 9 first."
model.load_state_dict(best_state)
model.eval()

tb_logits, tb_labels = [], []
ti_logits, ti_labels = [], []
tx_logits, tx_labels = [], []
tb_spk_masks = []
per_sample = []

with torch.no_grad():
    for batch in test_loader:
        input_ids         = batch["input_ids"].to(device)
        attention_mask    = batch["attention_mask"].to(device)
        b_labels_gpu      = batch["labels"].to(device)
        i_labels_gpu      = batch["intonation_labels"].to(device)
        x_labels_gpu      = batch["break_idx_labels"].to(device)

        out = model(input_ids, attention_mask)
        b_preds = out["boundary_logits"].argmax(dim=-1)
        i_preds = out["intonation_logits"].argmax(dim=-1)
        x_preds = out["break_idx_logits"].argmax(dim=-1)

        tb_spk_masks.append(batch["spk_mask"])
        tb_logits.append(out["boundary_logits"])
        tb_labels.append(b_labels_gpu)
        ti_logits.append(out["intonation_logits"])
        ti_labels.append(i_labels_gpu)
        tx_logits.append(out["break_idx_logits"])
        tx_labels.append(x_labels_gpu)

        for sid, tokens, bp, bl, ip, il, xp, xl in zip(
            batch["sample_ids"],
            batch["tokens"],
            b_preds, b_labels_gpu,
            i_preds, i_labels_gpu,
            x_preds, x_labels_gpu,
        ):
            per_sample.append({
                "sample_id":        sid,
                "tokens":           tokens,
                "boundary_preds":   bp[bl != -100].cpu().tolist(),
                "boundary_true":    bl[bl != -100].cpu().tolist(),
                "intonation_preds": ip[il != -100].cpu().tolist(),
                "intonation_true":  il[il != -100].cpu().tolist(),
                "break_idx_preds":  xp[xl != -100].cpu().tolist(),
                "break_idx_true":   xl[xl != -100].cpu().tolist(),
            })

fp_b, fl_b = flatten_predictions(tb_logits, tb_labels, all_spk_masks=tb_spk_masks)
fp_i, fl_i = flatten_predictions(ti_logits, ti_labels)
fp_x, fl_x = flatten_predictions(tx_logits, tx_labels)
test_metrics = {
    "boundary":   compute_metrics(fp_b, fl_b),
    "intonation": compute_multiclass_metrics(fp_i, fl_i),
    "break_idx":  compute_multiclass_metrics(fp_x, fl_x),
}

print("=" * 55)
print(f"  TEST SET  (best checkpoint — epoch {best_epoch})")
print("=" * 55)
b = test_metrics["boundary"]
print(f"  Boundary   F1={b['f1']:.4f}  P={b['precision']:.4f}  R={b['recall']:.4f}"
      f"  {'✓ beats baseline!' if b['f1'] > 0.77 else '✗ below 0.77 baseline'}")
i = test_metrics["intonation"]
print(f"  Intonation F1={i['f1']:.4f}  (macro, 3-class: rising/falling/level)")
x = test_metrics["break_idx"]
print(f"  Break idx  F1={x['f1']:.4f}  (macro, 2-class: index-3/index-4)")
print("=" * 55)

# ── Plot 1: Loss + boundary F1 curves ────────────────────────────────────────
epochs_ax = list(range(1, NUM_EPOCHS + 1))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.plot(epochs_ax, history["train_loss"], label="Train")
ax.plot(epochs_ax, history["val_loss"],   label="Val", linestyle="--")
ax.axvline(best_epoch, color="grey", linestyle=":", linewidth=1,
           label=f"Best (ep {best_epoch})")
ax.set_xlabel("Epoch"); ax.set_ylabel("Combined loss")
ax.set_title("Loss"); ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(epochs_ax, history["train_boundary_f1"],   label="Train boundary F1")
ax.plot(epochs_ax, history["val_boundary_f1"],     label="Val boundary F1",   linestyle="--")
ax.plot(epochs_ax, history["val_intonation_f1"],   label="Val intonation F1", linestyle=":")
ax.plot(epochs_ax, history["val_break_idx_f1"],    label="Val break idx F1",  linestyle="-.")
ax.axhline(0.77, color="red", linestyle=":", linewidth=1.2, label="Baseline F1 = 0.77")
ax.axvline(best_epoch, color="grey", linestyle=":", linewidth=1)
ax.set_xlabel("Epoch"); ax.set_ylabel("F1")
ax.set_title("F1 by head"); ax.legend(fontsize=8); ax.grid(alpha=0.3); ax.set_ylim(0, 1)

fig.suptitle(f"{RUN_ID}", fontsize=8, color="grey")
plt.tight_layout()
plt.savefig("/tmp/curves.png", dpi=120, bbox_inches="tight")
plt.show()

# ── Plot 2: Boundary confusion matrix ────────────────────────────────────────
cm  = confusion_matrix(fl_b, fp_b, labels=[0, 1])
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(cm, display_labels=["non-boundary", "boundary"]).plot(
    ax=ax, colorbar=False, cmap="Blues")
ax.set_title(f"Boundary — epoch {best_epoch}\n{RUN_ID}", fontsize=8)
plt.tight_layout()
plt.savefig("/tmp/confusion_matrix.png", dpi=120, bbox_inches="tight")
plt.show()

---
## Section 11 · Save Run

Saves to `runs/{RUN_ID}/`:

| File | Contents |
|---|---|
| `checkpoint/` | DistilBERT weights + tokenizer (HuggingFace format) |
| `hparams.json` | All hyperparameters — model, training, data split, **and annotation tool settings** from `meta.json` |
| `test_predictions.json` | Per-sample word-level predictions + true labels |
| `curves.png` | Loss + F1 learning curves |
| `confusion_matrix.png` | Test-set confusion matrix |


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 11 — Save checkpoint, hparams.json, predictions, plots ║
# ╚══════════════════════════════════════════════════════════════╝

os.makedirs(RUN_DIR, exist_ok=True)
print(f"Saving to: {RUN_DIR}\n")

# ── 1. Model checkpoint ───────────────────────────────────────────────────────
ckpt_dir = os.path.join(RUN_DIR, "checkpoint")
model.save_pretrained(ckpt_dir)
tokenizer.save_pretrained(ckpt_dir)
print("  ✓ checkpoint/    — model weights + tokenizer")

# ── 2. Hyperparameter provenance JSON ─────────────────────────────────────────
hparam_record = {
    "run_id":    RUN_ID,
    "run_notes": RUN_NOTES,
    "timestamp": datetime.now().isoformat(),

    "data": {
        "labels_root":     LABELS_ROOT,
        "n_samples_total": len(all_samples),
        "n_train":         len(train_ds),
        "n_val":           len(val_ds),
        "n_test":          len(test_ds),
        "split_seed":      SPLIT_SEED,
        "train_frac":      TRAIN_FRAC,
        "val_frac":        VAL_FRAC,
    },

    "preprocessing": {
        "strip_punctuation": STRIP_PUNCTUATION,
        "extra_features":    EXTRA_FEATURES,
        "max_length":        MAX_LENGTH,
    },

    "model": {
        "base":   MODEL_NAME,
        "heads":  {
            "boundary":   {"num_classes": 2, "labels": ["non-boundary", "boundary"]},
            "intonation": {"num_classes": 3, "labels": ["rising", "falling", "level"],
                           "note": "none(0) and unclear(4) masked to -100"},
            "break_idx":  {"num_classes": 2, "labels": ["index-3", "index-4"],
                           "note": "unannotated ('') masked to -100"},
        },
    },

    "training": {
        "batch_size":             BATCH_SIZE,
        "learning_rate":          LEARNING_RATE,
        "num_epochs":             NUM_EPOCHS,
        "warmup_steps":           WARMUP_STEPS,
        "weight_decay":           WEIGHT_DECAY,
        "imbalance_strategy":     IMBALANCE_STRATEGY,
        "boundary_loss_weight":   BOUNDARY_LOSS_WEIGHT,
        "boundary_task_weight":   BOUNDARY_TASK_WEIGHT,
        "intonation_task_weight": INTONATION_TASK_WEIGHT,
        "break_idx_task_weight":  BREAK_IDX_TASK_WEIGHT,
        "best_epoch":             best_epoch,
    },

    "results": {
        "best_val_boundary_f1": best_val_f1,
        "test": test_metrics,
        "val_history": {
            "loss":           history["val_loss"],
            "boundary_f1":    history["val_boundary_f1"],
            "boundary_prec":  history["val_boundary_prec"],
            "boundary_rec":   history["val_boundary_rec"],
            "intonation_f1":  history["val_intonation_f1"],
            "break_idx_f1":   history["val_break_idx_f1"],
        },
        "train_history": {
            "loss":           history["train_loss"],
            "boundary_f1":    history["train_boundary_f1"],
            "intonation_f1":  history["train_intonation_f1"],
            "break_idx_f1":   history["train_break_idx_f1"],
        },
    },

    "annotation_tool_meta": label_meta,
}

with open(os.path.join(RUN_DIR, "hparams.json"), "w") as f:
    json.dump(hparam_record, f, indent=2, cls=_NumpyEncoder)
print("  ✓ hparams.json")

# ── 3. Per-sample predictions ─────────────────────────────────────────────────
with open(os.path.join(RUN_DIR, "test_predictions.json"), "w") as f:
    json.dump(per_sample, f, indent=2)
print(f"  ✓ test_predictions.json  ({len(per_sample)} samples)")

# ── 4. Diagnostic plots ───────────────────────────────────────────────────────
for fname in ["curves.png", "confusion_matrix.png"]:
    shutil.copy(f"/tmp/{fname}", os.path.join(RUN_DIR, fname))
    print(f"  ✓ {fname}")

print(f"\n✓ Run {RUN_ID} fully saved.")
print(f"  {RUN_DIR}/")
print(f"  ├── checkpoint/")
print(f"  ├── hparams.json")
print(f"  ├── test_predictions.json")
print(f"  ├── curves.png")
print(f"  └── confusion_matrix.png")